# GT-04i — Équilibre corrélé (Aumann 1974) : le médiateur bat le Nash

**Série GameTheory** — famille 04 (Nash / formes normales).

Dans [GT-04](GameTheory-04-NashEquilibrium-Python.ipynb) on a vu que les équilibres de
Nash d'un jeu de coordination peuvent être *inefficients* : les joueurs se coordinent
sur l'équilibre « mauvais » et n'osent pas s'en écarter, car aucun ne gagne à changer
**unilatéralement**. L'équilibre **corrélé** (Aumann, 1974) brise ce verrou : un
**médiateur** commun tire un profil d'actions selon une distribution μ et recommande
*en privé* à chaque joueur quelle action jouer. Si, sachant sa recommandation, aucun
joueur ne gagne à dévier, μ est un équilibre corrélé.

L'idée clé, que ce carnet **mesure** sur un jeu 2×2 :

- l'ensemble des équilibres corrélés est un **polytope** — on le calcule en LP ;
- la corrélation permet d'atteindre un payoff que **aucun** Nash n'atteint ;
- le Nash est un cas particulier du corrélé (les NE sont des CE « dégénérés »).


## 1. Le jeu : Battle of the Sexes

Deux joueurs, deux actions chacune. La première (M) préfère l'Opéra, le second (J)
préfère le Football ; mais ils préfèrent **se retrouver ensemble** plutôt que de se
séparer.

|            | J: F | J: O |
|------------|------|------|
| **M: F**   | 3, 1 | 0, 0 |
| **M: O**   | 0, 0 | 1, 3 |

Payoffs (M, J). Les deux équilibres de Nash purs sont **(F,F)** → (3,1) et
**(O,O)** → (1,3) : coordonnés, mais *inéquitable* — l'un des deux « perd ».


In [1]:
import numpy as np

# Battle of the Sexes — payoffs (M, J), actions [F, O]
# index 0 = F, index 1 = O
A = np.array([[3, 0],
              [0, 1]])   # payoff de M  (lignes = action M, colonnes = action J)
B = np.array([[1, 0],
              [0, 3]])   # payoff de J
actions = ["F", "O"]
print("Payoff M (lignes=M, colonnes=J):")
print(A)
print("Payoff J:")
print(B)


Payoff M (lignes=M, colonnes=J):
[[3 0]
 [0 1]]
Payoff J:
[[1 0]
 [0 3]]


### Lecture

Le jeu est bien défini : `A[i,j]` est le payoff de M quand M joue `i` et J joue `j`.
Les coins (F,F) et (O,O) sont les deux Nash purs — on les retrouve explicitement à
la section 3.


## 2. Qu'est-ce qu'un équilibre corrélé ?

μ est une matrice de probabilités sur les paires d'actions (ici 2×2). Le médiateur
tire `(i,j) ~ μ` puis dit à M « joue `i` » et à J « joue `j` » (chacun ne voit que sa
propre recommandation).

μ est un **équilibre corrélé** si, pour chaque joueur, chaque recommandation, et
chaque action de déviation, *obéir* vaut au moins autant que *dévier* — **conditionnel
à la recommandation reçue** :

> pour tout `i, i'` (actions de M) :  Σⱼ μ(i,j) · [ A(i,j) − A(i',j) ] ≥ 0
> pour tout `j, j'` (actions de J) :  Σᵢ μ(i,j) · [ B(i,j) − B(i,j') ] ≥ 0

C'est un ensemble de **contraintes linéaires** en μ (plus Σμ = 1 et μ ≥ 0) → un
**polytope convexe**. `scipy.optimize.linprog` le manipule.


In [2]:
from scipy.optimize import linprog


def ce_constraints(A, B):
    """Inegalites de non-deviation de tout CE, en variables d = vec(mu)
    = [mu00, mu01, mu10, mu11]. Forme a.d <= b (pour linprog)."""
    m, n = A.shape
    ineq, const = [], []
    # Joueur M : pour chaque i, i' (i != i')
    for i in range(m):
        for ip in range(m):
            if i == ip:
                continue
            row = np.zeros(m * n)
            for j in range(n):
                row[i * n + j] = A[i, j] - A[ip, j]
            ineq.append(-row)
            const.append(0.0)
    # Joueur J : pour chaque j, j' (j != j')
    for j in range(n):
        for jp in range(n):
            if j == jp:
                continue
            row = np.zeros(m * n)
            for i in range(m):
                row[i * n + j] = B[i, j] - B[i, jp]
            ineq.append(-row)
            const.append(0.0)
    return np.array(ineq), np.array(const)


def all_ce_lp(A, B, objective=None):
    """LP complet des CE : non-deviation + sum(mu)=1 + mu>=0.
    `objective` (vec m*n) = payoff a MAXIMISER ; None = simple feasibility."""
    m, n = A.shape
    ineq, const = ce_constraints(A, B)
    c = np.zeros(m * n) if objective is None else -objective
    return linprog(c, A_ub=ineq, b_ub=const,
                   A_eq=np.ones((1, m * n)), b_eq=[1.0],
                   bounds=[(0, None)] * (m * n), method="highs")


def is_ce(mu, A, B, tol=1e-8):
    """Vrai si mu est un equilibre corrélé (toutes non-déviations satisfaites)."""
    ineq, _ = ce_constraints(A, B)
    d = np.asarray(mu, dtype=float).reshape(-1)
    return bool(np.all(ineq @ d <= tol)) and abs(d.sum() - 1) < tol and (d >= -tol).all()


def expected_payoff(mu, A, B):
    """Payoff en attente (M, J) sous la distribution jointe mu."""
    return (mu * A).sum(), (mu * B).sum()


ineq, const = ce_constraints(A, B)
print("Inegalites de non-deviation :", ineq.shape[0],
      "  (M : F->O, O->F   ;  J : F->O, O->F)")


Inegalites de non-deviation : 4   (M : F->O, O->F   ;  J : F->O, O->F)


### Lecture

On dispose des quatre briques : `ce_constraints` (les inégalités de non-dévi­ation),
`all_ce_lp` (le solveur sur le polytope), `is_ce` (test d'appartenance) et
`expected_payoff` (payoff en attente). Le jeu 2×2 donne **4** inégalités de
non-dévi­ation — deux par joueur.


In [3]:
# CE a payoff social maximal : objective = sum_ij mu_ij (A_ij + B_ij)
welfare = (A + B).reshape(-1)
res = all_ce_lp(A, B, objective=welfare)
mu_ce = res.x.reshape(A.shape)
print("Statut LP :", res.status, "-", res.message)
print("CE a payoff social maximal :", np.round(mu_ce, 3).tolist())
print("  support :", [(actions[i], actions[j], round(float(mu_ce[i, j]), 3))
                     for i in range(2) for j in range(2) if mu_ce[i, j] > 1e-6])
print("  payoff social maximal :", round(float(-res.fun), 3))


Statut LP : 0 - Optimization terminated successfully. (HiGHS Status 7: Optimal)
CE a payoff social maximal : [[1.0, 0.0], [0.0, 0.0]]
  support : [('F', 'F', 1.0)]
  payoff social maximal : 4.0


### Lecture

Le LP renvoie un **CE à payoff social maximal** : ici 4, atteint par toute
distribution **supportée sur les deux coins coordonnés** (F,F) et (O,O) — la masse
hors-diagonale (les issues « séparées ») est nulle. C'est toute une *face* du
polytope. Le **médiateur équitable** (1/2, 1/2) en est le milieu : même payoff social,
mais des parts égales. On le mesure juste après.


In [4]:
# Le mediateur equitable : 1/2 en (F,F), 1/2 en (O,O)
mu_mediator = np.array([[0.5, 0.0],
                        [0.0, 0.5]])
print("Le mediateur (1/2 F-F, 1/2 O-O) est un CE ?", is_ce(mu_mediator, A, B))
pM, pJ = expected_payoff(mu_mediator, A, B)
print(f"  -> payoff : M={pM:.3f}, J={pJ:.3f}   (payoff social {pM + pJ:.3f})")


Le mediateur (1/2 F-F, 1/2 O-O) est un CE ? True
  -> payoff : M=2.000, J=2.000   (payoff social 4.000)


### Lecture

Le médiateur donne **(2.0 ; 2.0)** avec un payoff social de 4 — le **maximum**
atteignable. C'est le chiffre-clé ; on le compare au Nash juste après.


## 3. Le Nash, mesuré — pour voir l'écart

On calcule le Nash **mixte** du même jeu (les purs sont les deux coins). L'écart
avec le CE n'est *mesurable* que si le Nash est mesuré de la même façon — d'où le
passage par `expected_payoff` sur la distribution jointe du Nash mixte.


In [5]:
# Nash mixte de BoS. Le jeu est ASYMETRIQUE : les deux seuils d'indifference
# sont DIFFERENTS (pas un seul q commun, comme pour un jeu symetrique).
#   M indifferant entre F et O :  3*q_J = 1*(1-q_J)   =>  q_J = 1/4
#   J indifferant entre F et O :  1*p_M = 3*(1-p_M)   =>  p_M = 3/4
pM_F = 3 / 4   # M joue F avec 3/4
pJ_F = 1 / 4   # J joue F avec 1/4
pM_dist = np.array([pM_F, 1 - pM_F])
pJ_dist = np.array([pJ_F, 1 - pJ_F])
mu_nash_mixed = np.outer(pM_dist, pJ_dist)
pM_nash, pJ_nash = expected_payoff(mu_nash_mixed, A, B)
print(f"Nash mixte : M joue F avec {pM_F:.3f}, J joue F avec {pJ_F:.3f}")
print(f"  -> M={pM_nash:.3f}, J={pJ_nash:.3f}")
print(f"  distribution jointe : {mu_nash_mixed.tolist()}")
print("  Nash purs : (F,F) -> (3,1)    et    (O,O) -> (1,3)")


Nash mixte : M joue F avec 0.750, J joue F avec 0.250
  -> M=0.750, J=0.750
  distribution jointe : [[0.1875, 0.5625], [0.0625, 0.1875]]
  Nash purs : (F,F) -> (3,1)    et    (O,O) -> (1,3)


### Interprétation — le tableau de bord

| Équilibre          | Payoff M | Payoff J |
|--------------------|----------|----------|
| Nash pur (F,F)     | 3.0      | 1.0      |
| Nash pur (O,O)     | 1.0      | 3.0      |
| Nash mixte         | 0.75     | 0.75     |
| **CE médiateur**   | **2.0**  | **2.0**  |

Le CE médiateur **(2;2)** domine **strictement** le Nash mixte **(0.75;0.75)** au sens
de Pareto — il améliore les deux joueurs à la fois. C'est le résultat central d'Aumann :
**la corrélation coordonnée par un médiateur atteint un accord que les joueurs ne
trouvent pas seuls** — et l'écart est d'autant plus net que le Nash mixte, lui,
égalise déjà les parts sans pouvoir les monter.


## 4. Où le Nash vit dans le polytope — et où il ne vit pas

Deux faits à **mesurer**, qui tracent la frontière exacte entre Nash et corrélé :

- les **Nash purs** (F,F) et (O,O) : des CE « dégénérés » (masse ponctuelle) — ils
  forment des **sommets** du polytope des équilibres corrélés ;
- le **Nash mixte**, vu comme distribution *indépendante* (produit des stratégies) :
  il est **lui aussi** un équilibre corrélé. Ce n'est pas un hasard : c'est le contenu
  du théorème d'Aumann — **tout équilibre de Nash est un équilibre corrélé** — que le
  carnet va *mesurer* avec le même `is_ce`.


In [6]:
# Les deux Nash purs sont des CE (les sommets du polytope) :
for (i, j, lab) in [(0, 0, "(F,F)"), (1, 1, "(O,O)")]:
    mu = np.zeros((2, 2)); mu[i, j] = 1.0
    print(f"Nash pur {lab} est un CE ?", is_ce(mu, A, B))

# Le Nash mixte EST un CE : tout equilibre de Nash est un equilibre correle (Aumann 1974).
# C'est le theoreme que ce carnet presente -- mesure ici sur la distribution du §3.
print("Nash mixte est un CE ?", is_ce(mu_nash_mixed, A, B),
      "   <- True : le Nash est INCLUS dans le corrélé")

# Controle negatif : la distribution UNIFORME sur les 4 profils est corrélée mais n'est
# PAS un CE -- la corrélation seule ne suffit pas, il faut la non-déviabilité conditionnelle.
mu_uniform = np.full((2, 2), 0.25)
print("Distribution uniforme est un CE ?", is_ce(mu_uniform, A, B),
      "   <- False : le polytope est strictement plus petit que le simplexe")


Nash pur (F,F) est un CE ? True
Nash pur (O,O) est un CE ? True
Nash mixte est un CE ? True    <- True : le Nash est INCLUS dans le corrélé
Distribution uniforme est un CE ? False    <- False : le polytope est strictement plus petit que le simplexe


### Lecture

La frontière est claire et **mesurée** :
- les **Nash purs** sont des CE — les sommets du polytope ;
- le **Nash mixte** est un CE aussi : c'est le théorème d'Aumann, « tout équilibre de
  Nash est un équilibre corrélé », vérifié par le même `is_ce` ;
- la distribution **uniforme**, elle, n'en est **pas** un — la corrélation seule ne
  fait pas un équilibre, il faut en plus la non-déviabilité conditionnelle. Le polytope
  des équilibres corrélés est donc **strictement plus petit** que le simplexe.

C'est là que la corrélation gagne : le médiateur **(2;2)** est un CE que **aucun**
équilibre de Nash n'atteint (ni les purs, ni le mixte à 0.75;0.75), et **aucun** CE
n'atteint (3;3). La corrélation n'est pas un détail : c'est ce qui sépare le médiateur
(2;2) du meilleur Nash (0.75;0.75).


## 5. Exercices

Trois exercices pour ancrer le mécanique. Chaque fonction est un **stub** à compléter
(`return None`) ; le carnet s'exécute de bout en bout même incomplet.


In [7]:
# Exercice 1 — Vérifieur de CE
# Compléter is_ce_custom : renvoyer True si mu est un équilibre corrélé.
# Indice : réutiliser ce_constraints(A, B) et tester  ineq @ mu.reshape(-1) <= 0.
def is_ce_custom(mu, A, B):
    # TODO etudiant
    return None
# Test attendu : is_ce_custom(mu_mediator, A, B) == True


In [8]:
# Exercice 2 — CE à payoff maximal pour un AUTRE jeu
# Soit le jeu de coordination ci-dessous. Compléter max_welfare_ce pour renvoyer la
# distribution mu (matrice 2x2) du CE à payoff social maximal.
# Indice : objective = (A2 + B2).reshape(-1), puis all_ce_lp(A2, B2, objective=...).
A2 = np.array([[4, 0], [0, 2]])
B2 = np.array([[2, 0], [0, 4]])


def max_welfare_ce(A2, B2):
    # TODO etudiant
    return None
# mu_etoile = max_welfare_ce(A2, B2)  -> support attendu sur les deux coins


In [9]:
# Exercice 3 — Lequel gagne ?
# Pour le jeu de base (A, B) : compléter ce_verdict pour renvoyer une phrase
# comparant le payoff du CE médiateur à celui du Nash mixte.
# Indice : expected_payoff(mu_mediator, A, B)  vs  (pM_nash, pJ_nash).
def ce_verdict(mu_mediator, A, B, pM_nash, pJ_nash):
    # TODO etudiant — ex : "Le CE mediateur (2.0; 2.0) domine le Nash mixte
    #                         (0.75; 0.75) pour J et ameliore M."
    return None


## En résumé

- L'équilibre **corrélé** est une distribution μ sur les profils d'actions que aucun
  joueur ne veut « trahir » sachant sa recommandation — un **polytope LP**.
- Sur un jeu de coordination, le médiateur atteint **(2;2)** là où le Nash mixte
  plafonne à **(0.75;0.75)** : la corrélation **crée** de la valeur.
- Le Nash est **inclus** dans le corrélé — purs et mixte y sont des CE, ce que `is_ce`
  mesure : c'est le théorème d'Aumann, un raffinement et non un remplacement.
- Le polytope est **strictement plus petit** que le simplexe : la distribution uniforme,
  corrélée, n'est pas un équilibre (mesuré) — la corrélation nécessite la
  non-déviabilité conditionnelle.

Prochaine étape de la série : le passage aux **jeux répétés** (GT-06), où la
corrélation prend la forme de **stratégies conditionnées à l'historique**.
